# Review sim trajectories

Loads a single sim output directory's raw tensors (`GtX.pt`/`GtV.pt`, plus `GtF.pt`/`GtC.pt`
when present), prints shapes/dtypes/particle counts, and plots a quick 3D scatter + a
trajectory summary (center-of-mass height and bounding-box extents over time) so you can
eyeball any `001a`-`005b` (or `011_...`) output dir without re-deriving the load/trim logic.

**Kernel:** use the `uniphy` conda env (`conda activate uniphy && python -m ipykernel install --user --name uniphy` once, if the kernel isn't registered yet) — it already has `torch`/`matplotlib`; no GPU/Taichi needed just to load and plot tensors.

**Trim conventions** (see `CLAUDE.md`'s "Important dt/frame gotcha" note and
`simulation/analyze_sim.py`, which this reuses): `dataset_generate.py`'s save loop calls
`get_x_gt(i, ...)` / `get_v_gt(i, ...)` / `get_C(i, ...)` (0-indexed) but `get_F(i+1, ...)`
(1-indexed) — so `GtX`/`GtV`/`GtC` are valid over `[0, n_valid)` while `GtF` (and `GtFtmp`/
`GtStress`, not loaded here) are valid over `[1, n_valid)`, where `n_valid = T - 2` (the last
two rows of every tensor are never written, leftover zero-init buffer).

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 -- registers the '3d' projection

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Point this at any simulation/output/<id> dir, e.g. one of the 10 canonical
# 001a-005b sims or the 011 probe-poke sim.
SIM_DIR = os.path.join(REPO_ROOT, "simulation", "output", "001a_elastic_sphere_tissue_free_fall_test_soft")

print("SIM_DIR:", SIM_DIR)
print("contents:", sorted(os.listdir(SIM_DIR)))

In [ ]:
def load_tensor(name):
    path = os.path.join(SIM_DIR, name)
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location="cpu")

X = load_tensor("GtX.pt")
V = load_tensor("GtV.pt")
F = load_tensor("GtF.pt")
C = load_tensor("GtC.pt")

for name, t in [("GtX", X), ("GtV", V), ("GtF", F), ("GtC", C)]:
    if t is None:
        print(f"{name}: not present")
    else:
        print(f"{name}: shape={tuple(t.shape)} dtype={t.dtype}")

assert X is not None and V is not None, "GtX.pt/GtV.pt are required"
num_substeps_total, num_particles, _ = X.shape
print(f"\nparticle count: {num_particles}")
print(f"raw substep rows (before trim): {num_substeps_total}")

In [ ]:
# Trim to the valid range per the conventions above.
n_valid = X.shape[0] - 2

X_np = X[:n_valid].numpy()
V_np = V[:n_valid].numpy()
F_np = F[1:n_valid].numpy() if F is not None else None
C_np = C[:n_valid].numpy() if C is not None else None

print(f"n_valid substeps: {n_valid}")
print(f"X_np: {X_np.shape}, V_np: {V_np.shape}")
if F_np is not None:
    print(f"F_np: {F_np.shape} (index 0 dropped)")
if C_np is not None:
    print(f"C_np: {C_np.shape}")

## 3D scatter: initial vs. final particle positions

In [ ]:
fig = plt.figure(figsize=(11, 5.5))

ax1 = fig.add_subplot(1, 2, 1, projection="3d")
p0 = X_np[0]
ax1.scatter(p0[:, 0], p0[:, 2], p0[:, 1], s=4, c=p0[:, 1], cmap="viridis")
ax1.set_title("Initial positions (substep 0)")
ax1.set_xlabel("x"); ax1.set_ylabel("z"); ax1.set_zlabel("y (up)")

ax2 = fig.add_subplot(1, 2, 2, projection="3d")
pN = X_np[-1]
ax2.scatter(pN[:, 0], pN[:, 2], pN[:, 1], s=4, c=pN[:, 1], cmap="viridis")
ax2.set_title(f"Final positions (substep {n_valid - 1})")
ax2.set_xlabel("x"); ax2.set_ylabel("z"); ax2.set_zlabel("y (up)")

fig.tight_layout()
plt.show()

## Trajectory summary: COM height + bounding-box extents over time

In [ ]:
com_y = X_np[:, :, 1].mean(axis=1)
extents = X_np.max(axis=1) - X_np.min(axis=1)
speed = np.linalg.norm(V_np, axis=2).mean(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(com_y)
axes[0].set_title("Center-of-mass height (y)")
axes[0].set_xlabel("substep"); axes[0].set_ylabel("y")
axes[0].grid(alpha=0.3)

axes[1].plot(extents[:, 0], label="extent_x")
axes[1].plot(extents[:, 1], label="extent_y")
axes[1].plot(extents[:, 2], label="extent_z")
axes[1].set_title("Bounding-box extents")
axes[1].set_xlabel("substep"); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(speed)
axes[2].set_title("Mean particle speed")
axes[2].set_xlabel("substep"); axes[2].set_ylabel("|v|")
axes[2].grid(alpha=0.3)

fig.tight_layout()
plt.show()

## Deformation magnitude (if `GtF.pt` is present)

Only meaningful for tensor materials (elasticity/von_mises/drucker_prager); `viscous_fluid`
only tracks a scalar volume ratio in `F[..., 0, 0]` (see `CLAUDE.md`).

In [ ]:
if F_np is not None:
    eye3 = np.eye(3, dtype=np.float32)
    deformation = np.linalg.norm(F_np - eye3, axis=(1, 2)).mean(axis=1)

    plt.figure(figsize=(6, 4))
    plt.plot(deformation)
    plt.title("Mean ||F - I|| over time")
    plt.xlabel("substep (offset by trimmed index 0)")
    plt.ylabel("deformation magnitude")
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("GtF.pt not present for this sim -- skipping.")